# Holonic Visualization Demo

Demonstrates `holonic.viz` using yFiles Jupyter Graphs:

1. **HolonViz** — single holon with layer grouping and colour coding
2. **HolarchyViz** — full holarchy with portal edges and optional expansion
3. **SPARQLExplorer** — interactive CONSTRUCT query widget with presets

In [ ]:
from holonic import Holon, TransformPortal, Holarchy, ProvenanceTracker
from holonic.viz import HolonViz, HolarchyViz, SPARQLExplorer

## Build a Holon with Rich Content

A municipal data holon with all four layers populated via TTL.

In [ ]:
vancouver = Holon(
    iri="urn:holon:city:vancouver",
    label="Vancouver",
    depth=2,

    interior_ttl="""
        @prefix muni: <urn:municipal:> .
        @prefix geo:  <urn:geo:> .

        <urn:city:vancouver> a geo:City, muni:CityRecord ;
            rdfs:label "Vancouver" ;
            muni:province "British Columbia" ;
            muni:country  "Canada" ;
            geo:population 675218 ;
            geo:latitude   49.2827 ;
            geo:longitude  -123.1207 .

        <urn:city:vancouver/param:area> a muni:MeasuredValue ;
            rdfs:label "area" ;
            muni:value 115.18 ;
            muni:unit  "km²" ;
            muni:of    <urn:city:vancouver> .
    """,

    boundary_ttl="""
        @prefix geo: <urn:geo:> .

        <urn:shapes:CityShape> a sh:NodeShape ;
            sh:targetClass geo:City ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ; sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "City must have exactly one label."
            ] ;
            sh:property [
                sh:path geo:population ;
                sh:minCount 1 ;
                sh:datatype xsd:integer ;
                sh:severity sh:Violation ;
                sh:message "Population is required."
            ] ;
            sh:property [
                sh:path geo:latitude ;
                sh:minCount 1 ;
                sh:datatype xsd:decimal ;
                sh:severity sh:Violation
            ] ;
            sh:property [
                sh:path geo:longitude ;
                sh:minCount 1 ;
                sh:datatype xsd:decimal ;
                sh:severity sh:Violation
            ] .
    """,

    projection_ttl="""
        <urn:holon:city:vancouver>
            cga:bindsTo <https://sws.geonames.org/6173331/> ;
            skos:exactMatch <https://www.wikidata.org/entity/Q24639> .
    """,

    context_ttl="""
        <urn:holon:city:vancouver>
            cga:memberOf <urn:holon:province:bc> .
    """,
)

print(vancouver)

## HolonViz — Single Holon View

Renders the holon's four named graphs as nested groups.
Nodes are colour-coded by layer:

| Layer | Colour | Shape |
|-------|--------|-------|
| Interior | Blue | Round Rectangle |
| Boundary | Purple | Hexagon |
| Projection | Green | Pill |
| Context | Amber | Octagon |

In [ ]:
# All four layers
viz = HolonViz(vancouver, layout="hierarchic")
w = viz.show()
w  # display in notebook

In [ ]:
# Interior + boundary only
viz_partial = HolonViz(vancouver, layers=["interior", "boundary"], layout="organic")
w2 = viz_partial.show()
w2

### With Interactive Controls

Toggle layers and layout algorithm from ipywidgets.

In [ ]:
viz.show_with_controls()

## Build a Holarchy

Three holons connected by transform portals.

In [ ]:
bc = Holon(
    iri="urn:holon:province:bc",
    label="British Columbia",
    depth=1,
    interior_ttl="""
        @prefix geo: <urn:geo:> .
        <urn:province:bc> a geo:Province ;
            rdfs:label "British Columbia" ;
            geo:population 5070000 ;
            geo:country "Canada" .
    """,
    boundary_ttl="""
        @prefix geo: <urn:geo:> .
        <urn:shapes:ProvinceShape> a sh:NodeShape ;
            sh:targetClass geo:Province ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:severity sh:Violation
            ] .
    """,
)

geonames = Holon(
    iri="urn:holon:consumer:geonames",
    label="GeoNames Consumer",
    depth=1,
    boundary_ttl="""
        @prefix geo: <urn:geo:> .
        <urn:shapes:FeatureShape> a sh:NodeShape ;
            sh:targetClass geo:Feature ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:severity sh:Violation
            ] ;
            sh:property [
                sh:path geo:population ;
                sh:minCount 1 ;
                sh:severity sh:Violation
            ] .
    """,
)

# Transform portal: Vancouver → GeoNames
portal = TransformPortal(
    iri="urn:portal:vancouver-to-geonames",
    source=vancouver,
    target=geonames,
    label="Municipal → GeoNames",
    construct_query="""
        PREFIX muni: <urn:municipal:>
        PREFIX geo:  <urn:geo:>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

        CONSTRUCT {
            ?city a geo:Feature .
            ?city rdfs:label ?name .
            ?city geo:population ?pop .
            ?city geo:latitude ?lat .
            ?city geo:longitude ?lon .
        }
        WHERE {
            ?city a geo:City .
            ?city rdfs:label ?name .
            ?city geo:population ?pop .
            OPTIONAL { ?city geo:latitude ?lat }
            OPTIONAL { ?city geo:longitude ?lon }
        }
    """,
)

# Record provenance
prov = ProvenanceTracker("urn:agent:demo", "Demo Agent")
prov.record_traversal(portal.iri, vancouver, geonames)

# Assemble holarchy
holarchy = Holarchy(label="Geographic Holarchy")
holarchy.register(bc)
holarchy.register(vancouver)
holarchy.register(geonames)
holarchy.add_portal(portal)

print(holarchy.summary())

## HolarchyViz — Collapsed View

Holons as single nodes, portals as edges.

In [ ]:
hviz = HolarchyViz(holarchy, show_internals=False, layout="hierarchic")
w3 = hviz.show()
w3

## HolarchyViz — Expanded View

Each holon expands to show its interior and boundary triples.

In [ ]:
hviz_expanded = HolarchyViz(
    holarchy,
    show_internals=True,
    layers=["interior", "boundary"],
    layout="hierarchic",
)
w4 = hviz_expanded.show()
w4

### With Interactive Controls

In [ ]:
hviz.show_with_controls()

## SPARQLExplorer — Interactive Query Widget

Execute SPARQL CONSTRUCT queries against the merged holarchy graph
and visualise the results live.

Includes:
- **Namespace manager** with auto-PREFIX generation
- **Preset projections**: All Triples, SHACL Shapes, Portal Network,
  Holarchy Structure, Provenance Trail, etc.
- **Custom queries**: write your own CONSTRUCT and see results immediately

In [ ]:
explorer = SPARQLExplorer(
    graph=holarchy.merged_all(),
    namespaces={
        "geo":  "urn:geo:",
        "muni": "urn:municipal:",
    },
    layout="hierarchic",
)
explorer.show()

## Programmatic Projection

Use the explorer's API directly to execute and visualise a query.

In [ ]:
result = explorer.execute("""
    PREFIX cga: <urn:cga:>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    CONSTRUCT {
        ?holon a cga:Holon .
        ?holon rdfs:label ?label .
        ?holon cga:hasPortal ?portal .
        ?portal cga:targetHolon ?target .
    }
    WHERE {
        ?holon a cga:Holon .
        ?holon rdfs:label ?label .
        OPTIONAL {
            ?holon cga:hasPortal ?portal .
            ?portal cga:targetHolon ?target .
        }
    }
""")

print(f"Result: {len(result)} triples")
w5 = explorer.visualise_result(result, layout="organic")
w5

## Colour Legend

| Visual | Meaning |
|--------|---------|
| 🔵 Blue nodes | Interior graph (A-Box data) |
| 🟣 Purple nodes / hexagons | Boundary graph (SHACL shapes) |
| 🟢 Green nodes / pills | Projection graph (external bindings) |
| 🟠 Amber nodes / octagons | Context graph (membership, temporal) |
| 🔴 Red triangles | Portals |
| ⬜ Light backgrounds | Group containers (holon / layer) |
| 🔘 Grey ellipses | Literal values |